# 第 1 節｜BERT 遮罩填空與中文情緒分類

**本節只使用預訓練或已微調好的模型進行推論，不自行訓練。** 對應 HTML 投影片第 1～6 頁。

- `google-bert/bert-base-chinese`：猜測 `[MASK]` 的 Token。
- `Johnson8187/Chinese-Emotion-Small`：預測八類情緒，屬於已微調的 mDeBERTa 模型。
- **全部 Code Cell 都已備妥程式，直接執行即可。** 只需修改句子、候選數等參數，觀察輸出差異。

## 準備環境｜先執行下方程式

在 Google Colab 選擇 GPU 可加快模型推論；第一次使用會下載模型權重。

In [ ]:
# Install the libraries for the two inference tasks.
# 安裝兩個文字推論任務需要的套件。
%pip -q install transformers==4.57.1 accelerate sentencepiece

# Import the tools for BERT inference and result tables.
# 匯入 BERT 推論與結果表格需要的工具。
import torch

# Import the text-processing pipeline.
# 匯入用來執行文字任務的 pipeline。
from transformers import pipeline

# Use the GPU when the Colab runtime provides one.
# Colab 提供 GPU 時優先使用 GPU。
device = 0 if torch.cuda.is_available() else -1

# Show the selected computing device.
# 顯示目前選用的運算裝置。
print("GPU available:", torch.cuda.is_available())


### 範例 1｜讓 BERT 完成遮罩填空

直接執行下一格程式，不必自己寫程式碼。可以修改：

- `MASK_SENTENCE`：換成想測試的句子，保留一個 `[MASK]`（代表一個 Token）。
- `TOP_K`：要顯示多少個候選結果（例如 3 或 5）。

這裡先顯示機率最高的填空句；下方 Optional 1 可列出多個候選。

In [ ]:
# Choose one sentence containing a masked token.
# 選一句包含遮罩 Token 的句子。
MASK_SENTENCE = "這道料理真[MASK]吃。"

# Set how many candidate tokens the model should predict.
# 設定模型要預測的候選 Token 數量。
TOP_K = 3

# Load the pretrained Chinese BERT mask-filling model.
# 載入預訓練的中文 BERT 遮罩填空模型。
mask_model = pipeline("fill-mask", model="google-bert/bert-base-chinese", device=device)

# Predict the missing token in the sentence.
# 預測句子中被遮住的 Token。
mask_results = mask_model(MASK_SENTENCE, top_k=TOP_K)

# Show the candidate sentence with the highest score.
# 顯示分數最高的填空句子。
print(mask_results[0]["sequence"])

### 認識 `pipeline()`：先選任務，再選模型

`pipeline()` 是 Hugging Face Transformers 提供的簡易**模型推論工具**。我們先選擇要做的任務（`task`），再指定能完成該任務的模型（`model`）；不用自己撰寫繁複的文字轉換和推論程式。

**請記住：任務和模型必須相容。** 不是每個 BERT 模型都能直接拿來做情緒分類。例如，預訓練的遮罩填空模型適合 `fill-mask`；要分類文字，應使用已具備文字分類能力的模型並搭配 `text-classification`。

**本節有兩種任務：**

- **遮罩填空：**`pipeline("fill-mask", model="google-bert/bert-base-chinese")`，讓模型猜 `[MASK]` 位置的文字。
- **情緒分類：**`pipeline("text-classification", model="Johnson8187/Chinese-Emotion-Small")`，讓已完成情緒分類訓練的模型判斷一句話的情緒。

**現在要做的是「文字分類」**：所以要選能進行文字分類的模型 `Johnson8187/Chinese-Emotion-Small`，`pipeline` 的任務也要設為 **`text-classification`**。

**Transformers 4.57.1 支援的 `pipeline()` 任務名稱：**

| 類型 | `task` 名稱 | 用途（簡單理解） |
|---|---|---|
| 文字 | `fill-mask` | 預測句子中 `[MASK]` 位置的文字 |
| 文字 | `text-classification` | 將整句文字分類，例如情緒或參加意願 |
| 文字 | `token-classification` | 替文字中的詞語分類，例如辨認人名與地名 |
| 文字 | `zero-shot-classification` | 用自己指定的候選類別分類文字 |
| 文字 | `question-answering` | 根據提供的文章回答問題 |
| 文字 | `summarization` | 將長文章整理成摘要 |
| 文字 | `translation` | 把文字翻譯成另一種語言 |
| 文字 | `translation_xx_to_yy` | 指定來源與目標語言的翻譯名稱格式，例如 `translation_en_to_fr` |
| 文字 | `text2text-generation` | 輸入文字，產生另一段文字 |
| 文字 | `text-generation` | 接續文字或產生新的文字內容 |
| 文字 | `feature-extraction` | 將文字轉成模型使用的數值特徵 |
| 影像／影片 | `image-classification` | 判斷圖片屬於哪個類別 |
| 影像／影片 | `object-detection` | 找出圖片中的物件與位置 |
| 影像／影片 | `image-segmentation` | 把圖片分成不同區域 |
| 影像／影片 | `depth-estimation` | 估計圖片中物體的遠近 |
| 影像／影片 | `image-to-text` | 根據圖片產生文字描述 |
| 影像／影片 | `image-to-image` | 由輸入圖片產生另一張圖片 |
| 影像／影片 | `image-feature-extraction` | 將圖片轉成數值特徵 |
| 影像／影片 | `mask-generation` | 產生圖片的區域遮罩 |
| 影像／影片 | `keypoint-matching` | 找出兩張圖片中對應的特徵點 |
| 影像／影片 | `video-classification` | 辨認影片的類別或活動 |
| 影像／影片 | `zero-shot-image-classification` | 依指定的候選類別分類圖片 |
| 影像／影片 | `zero-shot-object-detection` | 依指定物件名稱尋找圖片中的物件 |
| 聲音 | `audio-classification` | 辨認聲音種類，例如音效類別 |
| 聲音 | `automatic-speech-recognition` | 將語音轉成文字 |
| 聲音 | `text-to-audio` | 把文字轉成語音或聲音 |
| 聲音 | `zero-shot-audio-classification` | 依指定的候選類別辨認聲音 |
| 跨模態／文件 | `image-text-to-text` | 結合圖片與文字輸入，產生文字回答 |
| 跨模態／文件 | `visual-question-answering` | 根據圖片回答問題 |
| 跨模態／文件 | `document-question-answering` | 閱讀文件圖片並回答問題 |
| 跨模態／文件 | `table-question-answering` | 根據資料表回答問題 |

**補充：**`sentiment-analysis` 是 `text-classification` 的別名；`ner` 是 `token-classification` 的別名；`text-to-speech` 是 `text-to-audio` 的別名。不同任務需要相容的模型，**不能只改 `task` 名稱，就期待同一個模型完成所有工作**。部分任務還需要圖片、聲音或專用模型。

資料來源：[Hugging Face Transformers 4.57.1 官方 Pipeline API](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines)（與本 Notebook 安裝版本相同）。不同版本的支援清單可能調整。

### 範例 2｜使用現成模型辨認情緒

直接執行下一格程式，不必自己寫程式碼。只要修改 `EMOTION_TEXT`，就能觀察其他句子的預測標籤及分數。

這個情緒分類模型和前一個遮罩填空模型不同；若結果顯示 `LABEL_0`～`LABEL_7`，請查看下方 Markdown 對照表，不需要建立 Python 標籤字典。

In [ ]:
# Load the model already fine-tuned for Chinese emotion classification.
# 載入已完成中文情緒微調的分類模型。
emotion_model = pipeline("text-classification", model="Johnson8187/Chinese-Emotion-Small", device=device)

# Choose one Taiwanese Traditional Chinese sentence to classify.
# 選一句臺灣繁體中文句子來進行情緒分類。
EMOTION_TEXT = "今天的活動真的讓我很開心！"

# Try an alternative sentence by replacing EMOTION_TEXT above.
# 可以將上面的 EMOTION_TEXT 換成以下其中一句進行練習。
# EMOTION_TEXT = "怎麼會發生這種事？"
# EMOTION_TEXT = "今天的心情很平靜。"

# Predict the emotion of the selected sentence.
# 預測所選句子的情緒。
result = emotion_model(EMOTION_TEXT)

# Show the original model label and its confidence score.
# 顯示模型原始標籤與對應分數。
print(result[0]["label"], round(result[0]["score"], 3))

## Optional｜`LABEL_0` 到 `LABEL_7` 是什麼意思？

有些模型輸出的不是中文，而是 `LABEL_0` 等數字代碼。**每個模型對代碼的定義可能不同，必須查該模型的模型卡。** 此情緒模型的對照如下，依模型作者提供的順序整理：

| 模型標籤 | 情緒 | 模型標籤 | 情緒 |
|---|---|---|---|
| LABEL_0 | 平淡語氣 | LABEL_4 | 悲傷語調 |
| LABEL_1 | 關切語調 | LABEL_5 | 疑問語調 |
| LABEL_2 | 開心語調 | LABEL_6 | 驚奇語調 |
| LABEL_3 | 憤怒語調 | LABEL_7 | 厭惡語調 |

表格供查詢，不必在 Python 中自行撰寫 `emotion_names` 對照字典。


## Optional 1｜看看遮罩填空的前三個候選

先執行前面的遮罩填空範例，再直接執行下方程式。可以改動句子或 `top_k` 的數值。

In [ ]:
# Compare the three highest-probability mask candidates.
# 比較機率最高的三個遮罩候選字詞。
for item in mask_model("這個蛋糕非常[MASK]吃。", top_k=3):
    # Show each completed sentence and its model score.
    # 顯示每一個填空句子與模型分數。
    print(item["sequence"], round(item["score"], 3))


## Optional 2｜一句話比較三個情緒候選

直接執行下方程式；修改 `text` 即可比較其他情緒句子。其他測試句保留在程式註解中，方便複製替換。

`LABEL_n` 的中文意義請對照上方 Markdown 表格。模型分數不保證分類正確。

In [ ]:
# Choose one sentence to compare several emotion predictions.
# 選一句話，觀察模型給出的多個情緒候選。
text = "我收到錄取通知，真的好開心！"

# Try these alternatives by copying one line to replace the text above.
# 可以複製下列其中一句，取代上面的文字自行練習。
# text = "怎麼會發生這種事？"
# text = "今天的心情很平靜。"

# Show the three highest-scoring labels without manually converting LABEL_N.
# 顯示分數最高的三個原始標籤，不必手動建立 LABEL_N 對照字典。
print(emotion_model(text, top_k=3))


**本節總結：** `fill-mask` 是猜被遮住的 Token；`text-classification` 是輸出已定義的類別。預訓練 Encoder 本身並不等於已完成情緒分類微調。